In [ ]:
from google.colab import drive

drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
for path in [
    PROJECT,
    PROJECT / "data",
    RAW,
    CACHE,
    RESULTS,
    NOTEBOOKS
]:
    print(path, "->", path.exists())

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/UNSW_Project")
RAW = PROJECT / "data" / "raw"
CACHE = PROJECT / "data" / "cache"
RESULTS = PROJECT / "results"

print("Raw directory:", RAW)
print()

for f in sorted(RAW.iterdir()):
    if f.is_file():
        print(f"{f.name:35} {f.stat().st_size / (1024**2):8.2f} MB")

In [ ]:
!pip -q install pyarrow

In [ ]:
import pandas as pd

FEATURE_FILE = RAW / "NUSW-NB15_features.csv"

features = pd.read_csv(
    FEATURE_FILE,
    encoding="latin-1"
)

print("Shape:", features.shape)
display(features.head(20))

In [ ]:
import pandas as pd

sample_file = RAW / "UNSW-NB15_1.csv"

sample = pd.read_csv(sample_file,
    header=None,
    nrows=5,
    encoding="latin-1"
)

print("Shape of first 5 rows:", sample.shape)
display(sample)

In [ ]:
print(sample.iloc[0].tolist())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# PATHS
# ============================================================

PROJECT = Path("/content/drive/MyDrive/UNSW_Project")
RAW = PROJECT / "data" / "raw"
CACHE = PROJECT / "data" / "cache"
RESULTS = PROJECT / "results"

CACHE.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)


# ============================================================
# EXPECTED UNSW-NB15 COLUMNS
# ============================================================

COLS = [
    "srcip", "sport", "dstip", "dsport", "proto", "state",
    "dur", "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss",
    "service", "Sload", "Dload", "Spkts", "Dpkts", "swin", "dwin",
    "stcpb", "dtcpb", "smeansz", "dmeansz", "trans_depth",
    "res_bdy_len", "Sjit", "Djit", "Stime", "Ltime", "Sintpkt",
    "Dintpkt", "tcprtt", "synack", "ackdat", "is_sm_ips_ports",
    "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login",
    "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm",
    "ct_src_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm",
    "ct_dst_src_ltm", "attack_cat", "Label"
]


CATEGORICAL = [
    "proto",
    "state",
    "service"
]


NON_FEATURES = {
    "srcip",
    "dstip",
    "sport",
    "dsport",
    "Stime",
    "Ltime",
    "attack_cat",
    "Label",
    "ts",
    "family",
    "source_file",
    "session"
}


# ============================================================
# LOAD ONE FILE
# ============================================================

def load_one_file(path):

    print(f"\nLoading {path.name} ...")

    df = pd.read_csv(
        path,
        header=None,
        names=COLS,
        dtype=str,
        encoding="latin-1",
        low_memory=False
    )

    print(f"  Raw rows: {len(df):,}")

    df["source_file"] = path.name

    return df


# ============================================================
# CLEAN ONE FILE
# ============================================================

def clean_one_file(df):

    df = df.copy()

    # --------------------------------------------------------
    # Numeric columns
    # --------------------------------------------------------

    protected = {
        "srcip",
        "dstip",
        "sport",
        "dsport",
        "attack_cat",
        "source_file"
    } | set(CATEGORICAL)

    numeric_columns = [
        c for c in df.columns
        if c not in protected
    ]

    for c in numeric_columns:
        df[c] = pd.to_numeric(
            df[c],
            errors="coerce"
        )

    # --------------------------------------------------------
    # Fields where blank means "not applicable"
    # --------------------------------------------------------

    for c in [
        "ct_flw_http_mthd",
        "is_ftp_login",
        "ct_ftp_cmd"
    ]:
        df[c] = df[c].fillna(0)

    # --------------------------------------------------------
    # Categorical normalization
    # --------------------------------------------------------

    for c in CATEGORICAL:
        df[c] = (
            df[c]
            .astype(str)
            .str.strip()
            .str.lower()
        )

    # --------------------------------------------------------
    # Attack family
    # --------------------------------------------------------

    fam = (
        df["attack_cat"]
        .astype(str)
        .str.strip()
    )

    # Treat common representations of missing attack category
    # as benign.

    fam = fam.replace(
        {
            "": "Benign",
            "nan": "Benign",
            "NaN": "Benign",
            "none": "Benign",
            "None": "Benign"
        }
    )

    fam = fam.str.replace(
        r"\s+",
        " ",
        regex=True
    )

    # Normalize known spelling variation
    fam = fam.replace(
        {
            "Backdoors": "Backdoor"
        }
    )

    df["family"] = fam

    # --------------------------------------------------------
    # Binary label
    # --------------------------------------------------------

    df["Label"] = pd.to_numeric(
        df["Label"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Timestamp
    # --------------------------------------------------------

    df["ts"] = pd.to_datetime(
        pd.to_numeric(
            df["Stime"],
            errors="coerce"
        ),
        unit="s",
        errors="coerce"
    )

    # --------------------------------------------------------
    # Remove invalid rows
    # --------------------------------------------------------

    validation_columns = [
        c for c in df.columns
        if c not in {
            "srcip",
            "dstip",
            "sport",
            "dsport",
            "attack_cat",
            "source_file",
            "family"
        }
    ]

    invalid = df[validation_columns].isna().any(axis=1)

    invalid_count = int(invalid.sum())

    print(f"  Invalid rows: {invalid_count:,}")

    df = df.loc[~invalid].copy()

    return df, invalid_count


# ============================================================
# LOAD + CLEAN ALL FILES
# ============================================================

all_frames = []

audit_rows = []

files = [
    RAW / "UNSW-NB15_1.csv",
    RAW / "UNSW-NB15_2.csv",
    RAW / "UNSW-NB15_3.csv",
    RAW / "UNSW-NB15_4.csv"
]

for path in files:

    if not path.exists():
        raise FileNotFoundError(
            f"Missing file: {path}"
        )

    raw_df = load_one_file(path)

    raw_rows = len(raw_df)

    cleaned_df, invalid_count = clean_one_file(
        raw_df
    )

    audit_rows.append({
        "source_file": path.name,
        "rows_loaded": raw_rows,
        "invalid_rows": invalid_count,
        "rows_after_cleaning": len(cleaned_df)
    })

    all_frames.append(cleaned_df)

    # Free memory
    del raw_df


# ============================================================
# COMBINE CLEANED FILES
# ============================================================

print("\nCombining cleaned files...")

df = pd.concat(
    all_frames,
    ignore_index=True
)

del all_frames

print(
    f"Combined cleaned rows: {len(df):,}"
)


# ============================================================
# SORT BY TIME
# ============================================================

df = df.sort_values(
    "ts",
    kind="mergesort"
).reset_index(drop=True)


# ============================================================
# DUPLICATE AUDIT
# ============================================================

feature_columns = [
    c for c in df.columns
    if c not in NON_FEATURES
]

print("\nFeature columns used for duplicate detection:")
print(feature_columns)

before_duplicates = len(df)

df = df.drop_duplicates(
    subset=feature_columns + ["Label", "family"],
    keep="first"
).reset_index(drop=True)

duplicates_removed = (
    before_duplicates - len(df)
)

print(
    f"\nDuplicates removed: {duplicates_removed:,}"
)


# ============================================================
# IDENTIFY TEMPORAL SESSIONS
# ============================================================

gaps = (
    df["ts"]
    .diff()
    .dt.total_seconds()
)

session_id = (
    gaps > 3600
).cumsum()

df["session"] = session_id


# ============================================================
# SESSION SUMMARY
# ============================================================

sessions = (
    df.groupby("session")
      .agg(
          start=("ts", "min"),
          end=("ts", "max"),
          rows=("ts", "size")
      )
      .reset_index()
)

sessions["duration_hours"] = (
    sessions["end"] - sessions["start"]
).dt.total_seconds() / 3600


# ============================================================
# FAMILY COUNTS
# ============================================================

family_counts = (
    df["family"]
    .value_counts()
    .rename_axis("family")
    .reset_index(name="count")
)


# ============================================================
# LABEL/FAMILY CONSISTENCY
# ============================================================

label_family_1 = int(
    (
        (df["Label"] == 0) &
        (df["family"] != "Benign")
    ).sum()
)

label_family_2 = int(
    (
        (df["Label"] == 1) &
        (df["family"] == "Benign")
    ).sum()
)


# ============================================================
# PER-FAMILY TIME RANGE
# ============================================================

family_timeline = (
    df.groupby("family")
      .agg(
          first_timestamp=("ts", "min"),
          last_timestamp=("ts", "max"),
          count=("ts", "size")
      )
      .reset_index()
)


# ============================================================
# FAMILY DISTRIBUTION BY SESSION
# ============================================================

session_family = pd.crosstab(
    df["session"],
    df["family"]
)


# ============================================================
# 20-BLOCK TEMPORAL COVERAGE
# ============================================================

coverage_rows = []

for session, g in df.groupby("session"):

    if len(g) < 20:
        continue

    time_values = (
        g["ts"].astype("int64")
    )

    blocks = pd.cut(
        time_values,
        bins=20,
        labels=False,
        duplicates="drop"
    )

    temp = g.assign(
        block=blocks
    )

    coverage = (
        temp.groupby("family")["block"]
            .nunique()
            .rename(f"session_{session}_blocks_present")
    )

    coverage_rows.append(coverage)


if coverage_rows:

    block_coverage = pd.concat(
        coverage_rows,
        axis=1
    ).fillna(0).astype(int)

else:

    block_coverage = pd.DataFrame()


# ============================================================
# LARGE TIME GAPS
# ============================================================

large_gaps = gaps[gaps > 3600]

gap_rows = []

for index, gap in large_gaps.items():

    gap_rows.append({
        "before_row": index - 1,
        "after_row": index,
        "gap_hours": gap / 3600,
        "before_time": df.loc[index - 1, "ts"],
        "after_time": df.loc[index, "ts"]
    })

large_gap_table = pd.DataFrame(
    gap_rows
)


# ============================================================
# AUDIT SUMMARY
# ============================================================

audit = {
    "rows_after_cleaning_before_dedup": before_duplicates,
    "duplicates_removed": duplicates_removed,
    "rows_final": len(df),
    "families": df["family"].nunique(),
    "sessions": df["session"].nunique(),
    "timestamp_min": df["ts"].min(),
    "timestamp_max": df["ts"].max(),
    "label0_family_nonbenign": label_family_1,
    "label1_family_benign": label_family_2
}

audit_df = pd.DataFrame([audit])


# ============================================================
# SAVE RESULTS
# ============================================================

pd.DataFrame(audit_rows).to_csv(
    RESULTS / "file_audit.csv",
    index=False
)

audit_df.to_csv(
    RESULTS / "audit_unsw.csv",
    index=False
)

family_counts.to_csv(
    RESULTS / "family_counts.csv",
    index=False
)

sessions.to_csv(
    RESULTS / "capture_sessions.csv",
    index=False
)

session_family.to_csv(
    RESULTS / "session_family_counts.csv"
)

family_timeline.to_csv(
    RESULTS / "family_timeline.csv",
    index=False
)

block_coverage.to_csv(
    RESULTS / "block_coverage.csv"
)

large_gap_table.to_csv(
    RESULTS / "large_time_gaps.csv",
    index=False
)


# ============================================================
# SAVE FULL CLEANED DATA
# ============================================================

cleaned_cache = (
    CACHE / "unsw_cleaned_full.parquet"
)

print(
    f"\nSaving cleaned dataset to:\n{cleaned_cache}"
)

df.to_parquet(
    cleaned_cache,
    index=False
)


# ============================================================
# PRINT IMPORTANT RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FILE AUDIT")
print("=" * 70)

display(
    pd.DataFrame(audit_rows)
)

print("\n")
print("=" * 70)
print("FAMILY COUNTS")
print("=" * 70)

display(family_counts)


print("\n")
print("=" * 70)
print("LABEL / FAMILY CONSISTENCY")
print("=" * 70)

print(
    "Label=0 but family != Benign:",
    label_family_1
)

print(
    "Label=1 but family == Benign:",
    label_family_2
)


print("\n")
print("=" * 70)
print("TIME RANGE")
print("=" * 70)

print(
    df["ts"].min(),
    "->",
    df["ts"].max()
)


print("\n")
print("=" * 70)
print("CAPTURE SESSIONS")
print("=" * 70)

display(sessions)


print("\n")
print("=" * 70)
print("LARGE TIME GAPS")
print("=" * 70)

display(large_gap_table)


print("\n")
print("=" * 70)
print("FAMILY TIMELINE")
print("=" * 70)

display(family_timeline)


print("\n")
print("=" * 70)
print("FAMILY DISTRIBUTION BY SESSION")
print("=" * 70)

display(session_family)


print("\n")
print("=" * 70)
print("20-BLOCK TEMPORAL COVERAGE")
print("=" * 70)

display(block_coverage)


print("\n")
print("=" * 70)
print("FINAL AUDIT")
print("=" * 70)

display(audit_df)

print("\nAudit complete.")
print("Cleaned dataset saved to:")
print(cleaned_cache)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# PATHS
# ============================================================
PROJECT = Path("/content/drive/MyDrive/UNSW_Project")
RAW = PROJECT / "data" / "raw"
CACHE = PROJECT / "data" / "cache"
RESULTS = PROJECT / "results"

CACHE.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# ============================================================
# EXPECTED UNSW-NB15 COLUMNS
# ============================================================
COLS = [
    "srcip", "sport", "dstip", "dsport", "proto", "state",
    "dur", "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss",
    "service", "Sload", "Dload", "Spkts", "Dpkts", "swin", "dwin",
    "stcpb", "dtcpb", "smeansz", "dmeansz", "trans_depth",
    "res_bdy_len", "Sjit", "Djit", "Stime", "Ltime", "Sintpkt",
    "Dintpkt", "tcprtt", "synack", "ackdat", "is_sm_ips_ports",
    "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login",
    "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm",
    "ct_src_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm",
    "ct_dst_src_ltm", "attack_cat", "Label"
]

CATEGORICAL = ["proto", "state", "service"]

NON_FEATURES = {
    "srcip", "dstip", "sport", "dsport", "Stime", "Ltime",
    "attack_cat", "Label", "ts", "family", "source_file", "session"
}

# Map columns to optimized datatypes to speed up load time
DTYPES = {}
for col in COLS:
    if col in ["srcip", "sport", "dstip", "dsport", "attack_cat", "Label"] + CATEGORICAL:
        DTYPES[col] = str
    else:
        DTYPES[col] = float # Use float to handle any NaNs natively

# ============================================================
# LOAD AND CLEAN ONE FILE (OPTIMIZED)
# ============================================================
def load_and_clean_file(path):
    print(f"\nLoading & cleaning {path.name} ...")

    # Read file using defined fast types, treating spaces and empty strings as NaN
    df = pd.read_csv(
        path,
        header=None,
        names=COLS,
        dtype=DTYPES,
        encoding="latin-1",
        low_memory=False,
        na_values=[" ", ""]
    )
    raw_rows = len(df)
    print(f"  Raw rows: {raw_rows:,}")

    # Clean categorical columns efficiently
    for c in CATEGORICAL:
        df[c] = df[c].fillna("").astype(str).str.strip().str.lower()

    # Fields where blank means "not applicable"
    for c in ["ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd"]:
        df[c] = df[c].fillna(0)

    # Attack family
    fam = df["attack_cat"].fillna("Benign").astype(str).str.strip()
    fam = fam.replace({"nan": "Benign", "NaN": "Benign", "none": "Benign", "None": "Benign", "": "Benign"})
    fam = fam.str.replace(r"\s+", " ", regex=True)
    fam = fam.replace({"Backdoors": "Backdoor"})
    df["family"] = fam

    # Binary label
    df["Label"] = pd.to_numeric(df["Label"], errors="coerce").fillna(0).astype(int)

    # Timestamp
    df["ts"] = pd.to_datetime(pd.to_numeric(df["Stime"], errors="coerce"), unit="s", errors="coerce")

    df["source_file"] = path.name

    # Remove invalid rows
    validation_columns = [c for c in df.columns if c not in {"srcip", "dstip", "sport", "dsport", "attack_cat", "source_file", "family"}]
    invalid = df[validation_columns].isna().any(axis=1)
    invalid_count = int(invalid.sum())
    print(f"  Invalid rows: {invalid_count:,}")

    df = df.loc[~invalid].copy()
    return df, raw_rows, invalid_count

# ============================================================
# RUN PIPELINE
# ============================================================
all_frames = []
audit_rows = []
files = [
    RAW / "UNSW-NB15_1.csv",
    RAW / "UNSW-NB15_2.csv",
    RAW / "UNSW-NB15_3.csv",
    RAW / "UNSW-NB15_4.csv"
]

for path in files:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    cleaned_df, raw_rows, invalid_count = load_and_clean_file(path)
    audit_rows.append({
        "source_file": path.name,
        "rows_loaded": raw_rows,
        "invalid_rows": invalid_count,
        "rows_after_cleaning": len(cleaned_df)
    })
    all_frames.append(cleaned_df)

print("\nCombining cleaned files...")
df = pd.concat(all_frames, ignore_index=True)
del all_frames
print(f"Combined cleaned rows: {len(df):,}")

# Sort by time
df = df.sort_values("ts", kind="mergesort").reset_index(drop=True)

# Deduplicate
feature_columns = [c for c in df.columns if c not in NON_FEATURES]
before_duplicates = len(df)
df = df.drop_duplicates(subset=feature_columns + ["Label", "family"], keep="first").reset_index(drop=True)
duplicates_removed = before_duplicates - len(df)
print(f"Duplicates removed: {duplicates_removed:,}")

# Save the final optimized dataframe
cleaned_cache = CACHE / "unsw_cleaned_full.parquet"
df.to_parquet(cleaned_cache, index=False)
print(f"Saved cleaned dataset to: {cleaned_cache}")

In [ ]:
import pandas as pd, numpy as np

df = pd.read_parquet(CACHE / "unsw_cleaned_full.parquet")
print("Shape:", df.shape)
print("Columns:", list(df.columns))

# Find the columns, whatever you named them
fam_col  = next(c for c in df.columns if c.lower() in ("family","attack_cat"))
lab_col  = next(c for c in df.columns if c.lower() == "label")
time_col = next(c for c in df.columns if c.lower() in ("stime","ts"))

# Build a sorted datetime column
if np.issubdtype(df[time_col].dtype, np.datetime64):
    df["ts"] = df[time_col]
else:
    df["ts"] = pd.to_datetime(df[time_col], unit="s")
df = df.sort_values("ts", kind="mergesort").reset_index(drop=True)

df["fam"] = df[fam_col].fillna("Benign").astype(str).str.strip()
df.loc[df["fam"].isin(["", "nan"]), "fam"] = "Benign"

print("\n=== FAMILY COUNTS ===")
print(df["fam"].value_counts().to_string())

print("\n=== LABEL/FAMILY CONSISTENCY (both should be ~0) ===")
print(((df[lab_col]==0) & (df.fam!="Benign")).sum(),
      ((df[lab_col]==1) & (df.fam=="Benign")).sum())

print("\n=== TIME RANGE ===")
print(df.ts.min(), "->", df.ts.max())
gaps = df.ts.diff().dt.total_seconds()
for i, g in gaps[gaps > 3600].items():
    print(f"gap before row {i}: {g/3600:.1f} h ({df.ts[i-1]} -> {df.ts[i]})")

print("\n=== PER-FAMILY first/last/count ===")
print(df.groupby("fam")["ts"].agg(["min","max","count"]).to_string())

# Block coverage: 20 equal-time blocks per session
df["session"] = (gaps > 3600).fillna(False).cumsum()
cols = []
for s, g in df.groupby("session"):
    blk = pd.cut(g.ts.astype("int64"), 20, labels=False)
    cols.append(g.assign(block=blk).groupby("fam")["block"].nunique()
                 .rename(f"sess{s}_blocks_of_20"))
print("\n=== BLOCK COVERAGE ===")
print(pd.concat(cols, axis=1).fillna(0).astype(int).to_string())

print("\n=== ROWS PER SESSION ===")
print(df.groupby("session").size())

In [ ]:
import os
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/UNSW_Project")
CACHE = PROJECT / "data" / "cache"

print("Cache directory exists:", CACHE.exists())
if CACHE.exists():
    print("Contents of cache directory:")
    for f in CACHE.iterdir():
        print(f"- {f.name} ({f.stat().st_size / (1024**2):.2f} MB)")
else:
    print("Creating Cache Directory...")
    CACHE.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# DUPLICATE + ATTACK FAMILY VERIFICATION
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_parquet(
    CACHE / "unsw_cleaned_full.parquet"
)

print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# 1. Check attack-family values exactly
# ------------------------------------------------------------

print("\n=== EXACT FAMILY VALUES ===")
print(
    df["family"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------
# 2. Check raw attack_cat values
# ------------------------------------------------------------

print("\n=== RAW attack_cat VALUES ===")
print(
    df["attack_cat"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------
# 3. Check Backdoor variants
# ------------------------------------------------------------

print("\n=== BACKDOOR VARIANTS ===")

backdoor_mask = (
    df["family"]
    .astype(str)
    .str.lower()
    .str.contains("backdoor", na=False)
)

print(
    df.loc[
        backdoor_mask,
        ["attack_cat", "family", "Label", "source_file"]
    ]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 4. Check duplicate counts AGAIN
# ------------------------------------------------------------

NON_FEATURES_AUDIT = {
    "srcip", "dstip", "sport", "dsport",
    "Stime", "Ltime",
    "attack_cat", "Label",
    "ts", "family",
    "source_file", "session"
}

duplicate_columns = [
    c for c in df.columns
    if c not in NON_FEATURES_AUDIT
]

print("\n=== DUPLICATE AUDIT ===")
print("Columns used:", len(duplicate_columns))

duplicate_mask = df.duplicated(
    subset=duplicate_columns + ["Label", "family"],
    keep=False
)

print(
    "Rows belonging to duplicate groups:",
    int(duplicate_mask.sum())
)

print(
    "Unique duplicate groups:",
    df.loc[
        duplicate_mask,
        duplicate_columns + ["Label", "family"]
    ].drop_duplicates().shape[0]
)

# ------------------------------------------------------------
# 5. Duplicate distribution by source file
# ------------------------------------------------------------

print("\n=== DUPLICATE ROWS BY SOURCE FILE ===")

print(
    df.loc[
        duplicate_mask,
        "source_file"
    ]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 6. Family distribution of duplicate rows
# ------------------------------------------------------------

print("\n=== DUPLICATE ROWS BY FAMILY ===")

print(
    df.loc[
        duplicate_mask,
        "family"
    ]
    .value_counts()
    .to_string()
)

print("\nVerification complete.")